## Extension 4 — EVA reproduction and reproducibility audit

This notebook reconstructs the paper's described EVA workflow using pinned SunSight data/configuration, its objective and projection interfaces, and NEAT-Python genome operations. It regenerates the two requested figure types. It does **not** numerically reproduce the published curves or establish that a released checkpoint is the exact figure-generating model.

Figure 3 compares Carbon Offset, Energy Generation, Racial Equity, and Income Equity with a reconstructed EIA status quo at each positive budget. Figure 4 maps the scoring network's outputs, not allocated installation counts.

### Objective definitions

The count-balance equity objectives after additions are, with $b_i=I_i+a_i$ representing existing-plus-new installations,

$$f_{IE}(a)=1-\frac{|\sum_{i\in H_I}b_i-\sum_{i\in L_I}b_i|}{\sum_i b_i},$$

$$f_{RE}(a)=1-\frac{|\sum_{i\in H_R}b_i-\sum_{i\in L_R}b_i|}{\sum_i b_i}.$$

This follows Section 6.2's description of evaluating the mix after new installations are factored in; an added-only equity calculation would be a different experiment. Median ties go into the high group. The linear objectives use the released toolkit indicators `carbon_offset_kg_per_panel` and `yearly_sunlight_kwh_kw_threshold_avg`. Ratios are reported; these legacy indicators and allocation caps are not reinterpreted as a physically verified individual-panel inventory.

$$CPR(a)=\sum_{k=1}^4\frac{f_k(a)}{f_k(a^{SQ})}.$$

CPR above four does not imply that every objective exceeds its baseline.

### Training

There are five measured attributes, min-max normalized: income, carbon, solar resource, stored panel utilization, and Black population proportion. A constant leading input reproduces the released six-input configuration's bias expansion. It is not a sixth measured attribute.

Population: 30; generations: six; training budget: two million; survival target: 0.3; one selected elite. For objective selectivities $v_k$, each sequential filter retains approximately

$$s_k=0.3^{v_k/\sum_jv_j}$$

of the current survivors. Objectives are randomly ordered. Survivors are finally ranked by CPR. Twenty-nine distinct selected-parent pairs generate mutated offspring, plus one elite. The best selected CPR checkpoint seen during the run is retained. The global parent-pool reconstruction does not reproduce NEAT-Population speciation or call the released binary-fitness training wrapper unchanged.

Adaptive selectivity starts at one for each objective and increases the weight of objectives below baseline. At most six tuning rounds are permitted; all three reported seeds passed at the first round. Seed 0 was designated as the primary figure before comparing seeds; seeds 1 and 2 are diagnostics.

### Baseline reconstruction

The supplementary CSV is transcribed from EIA's May 2025 Electric Power Monthly, Table 6.2.B: March 2025 and March 2024 estimated small-scale PV capacity, all sectors. The paper specifies March-to-March, whereas the released monthly CSV stops at February 2025 and its helper uses January residential capacity. The precise historical extraction underlying the figures is not identified.

Negative net changes (Rhode Island and Missouri) are clipped to zero. Alabama is marked NM (not meaningful due to large relative standard error) and is excluded rather than imputed. State growth shares are combined with within-state existing-installation shares and redistributed within states where caps bind. This yields 10,243 eligible ZIP codes. These are declared reconstruction choices, not an assertion that the authors made identical choices.

The legacy SunSight `count_qualified` cap is retained for this separate reproduction experiment. It differs from Question 2's estimated individual-panel capacity. There is no claim that the numerical energy or capacity units match Questions 2–3.

### Metric drift and reproduction status

The released `calc_equity` implementation uses existing-plus-new installation density per potential panel and a formula of the form $2-|\bar u_H-\bar u_L|/\bar u$, not the count-balance equations in paper Section 4.2. Those metrics must not silently be interchanged.

The selected released checkpoint is `Models/Neat/models/05-31-25/NEAT_model_lexicase_weighted_5.pkl`. Its name does not establish that it generated the paper's figures. Replaying it in this reconstructed experiment does not reproduce four-objective dominance.

For fresh training, seed 0 has endpoint ratios approximately (1.2880, 1.1944, 1.0218, 1.3901). It exceeds status quo on all objectives at only seven of twenty evaluated budgets. The paper's all-budget dominance is therefore not reproduced here.

The reconstructed racial-equity baseline at two million is approximately 0.8802. Since the count-balance objective is bounded by one, the maximum possible ratio under these conditions is $1/0.8802\approx1.1361$. The paper's approximately 1.24 reported racial-equity ratio cannot be matched under this exact reconstructed denominator. This shows that the evaluation conditions differ; it does not by itself establish an error in the paper.

Raw sigmoid scores concentrate near one. They are ranking values, not probabilities. The figure shows raw scores plus a clearly labeled percentile diagnostic; percentile coloring does not change allocations.

### Model inspection

The primary seed-0 network contains one sigmoid output and no hidden nodes. Its ranking is therefore equivalent to a linear combination of the retained normalized inputs. The experiment does not demonstrate an advantage from learned nonlinear ranking structure. The equity metrics compare ZIP groups, not the reported race or income of individual solar owners.

### References

- Paper 2, Sections 4.2–5.2 and 7.1: https://doi.org/10.1145/3736425.3770115
- Pinned toolkit: https://github.com/coopersigrist/SunSight/tree/99588d1696c980075a19abf087c669ae5aa22e4e
- Archived EIA input, Table 6.2.B: https://www.eia.gov/electricity/monthly/archive/may2025.pdf
- NEAT-Python: https://neat-python.readthedocs.io/


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "Utils"))
from paths import DATA, NEAT_CONFIG, SUNSIGHT

In [ ]:
import copy
import itertools
import random
from pathlib import Path

import neat
import numpy as np
import pandas as pd

from Simulation.projections_util import Objective, Projection


FIELDS = ["Median_income", "carbon_offset_kg_per_panel", "yearly_sunlight_kwh_kw_threshold_avg", "panel_utilization", "black_prop"]
OBJECTIVES = ["Carbon", "Energy", "Racial equity", "Income equity"]


def linear_objective(frame, allocation, column):
    return float(allocation @ frame[column].to_numpy())


def equity_objective(frame, allocation, field):
    high = frame[field].ge(frame[field].median()).to_numpy()
    installed = allocation + frame["existing_installs_count"].to_numpy()
    return 1 - abs(installed[high].sum() - installed[~high].sum()) / installed.sum()


def prepare_experiment(raw, eia):
    complete = eia.dropna(subset=["March2025", "March2024"]).set_index("State")
    data = raw.loc[raw["state_name"].isin(complete.index)].sort_values("region_name").reset_index(drop=True).copy()
    required = FIELDS + ["count_qualified", "existing_installs_count"]
    if not np.isfinite(data[required].to_numpy()).all() or not data["count_qualified"].gt(0).all():
        raise ValueError("Required EVA fields must be finite, with positive capacities.")
    if data["region_name"].duplicated().any():
        raise ValueError("ZIP identifiers must be unique.")
    values = data[FIELDS].to_numpy()
    spread = values.max(axis=0) - values.min(axis=0)
    inputs = np.column_stack([np.ones(len(data)), (values - values.min(axis=0)) / np.where(spread > 0, spread, 1)])
    growth = (complete["March2025"] - complete["March2024"]).clip(lower=0)
    state_totals = data.groupby("state_name")["existing_installs_count"].sum()
    within_state = data["existing_installs_count"] / data["state_name"].map(state_totals).replace(0, np.nan)
    weights = (data["state_name"].map(growth) * within_state).fillna(0).to_numpy()
    if weights.sum() <= 0:
        raise ValueError("The EIA baseline must have positive placement mass.")
    weights /= weights.sum()
    objectives = [Objective("Carbon", linear_objective, column=FIELDS[1]), Objective("Energy", linear_objective, column=FIELDS[2]), Objective("Racial equity", equity_objective, field="black_prop"), Objective("Income equity", equity_objective, field="Median_income")]
    return {"data": data, "inputs": inputs, "capacity": np.floor(data["count_qualified"]).to_numpy(dtype=np.int64), "weights": weights, "groups": data.groupby("state_name").indices, "objectives": objectives}


def evaluate(experiment, allocation):
    return np.array([objective.calc(experiment["data"], allocation) for objective in experiment["objectives"]])


def rank_allocation(scores, capacities, n):
    if n <= 0 or n > capacities.sum():
        raise ValueError("The installation budget must be positive and feasible.")
    order = np.argsort(-scores, kind="stable")
    remaining = n - np.concatenate(([0], np.cumsum(capacities[order][:-1])))
    allocation = np.zeros(len(capacities), dtype=np.int64)
    allocation[order] = np.minimum(capacities[order], np.maximum(remaining, 0))
    return allocation


def proportional_with_caps(weights, capacities, n):
    allocation = np.zeros(len(weights))
    active = (weights > 0) & (capacities > 0)
    if n > capacities[active].sum() + 1e-7:
        raise ValueError("The state-level baseline budget exceeds eligible capacity.")
    remaining = float(n)
    while remaining > 1e-7:
        shares = remaining * weights[active] / weights[active].sum()
        allocation[active] += np.minimum(shares, capacities[active] - allocation[active])
        remaining = n - allocation.sum()
        active = (weights > 0) & (capacities - allocation > 1e-8)
    return allocation


def baseline_allocation(experiment, n):
    allocation = np.zeros(len(experiment["data"]))
    for indices in experiment["groups"].values():
        weights = experiment["weights"][indices]
        amount = n * weights.sum()
        if amount > 0:
            allocation[indices] = proportional_with_caps(weights, experiment["capacity"][indices], amount)
    return allocation


def network_scores(network, inputs):
    values = {node: inputs[:, i] for i, node in enumerate(network.input_nodes)}
    values.update({node: np.zeros(len(inputs)) for node in network.output_nodes})
    for node, activation, aggregation, bias, response, links in network.node_evals:
        if aggregation.__name__ != "sum_aggregation":
            raise ValueError("This evaluator requires the configured sum aggregation.")
        total = np.zeros(len(inputs))
        for source, weight in links:
            total += weight * values[source]
        z = bias + response * total
        if activation.__name__ == "sigmoid_activation":
            output = 1 / (1 + np.exp(-np.clip(5 * z, -60, 60)))
        elif activation.__name__ == "relu_activation":
            output = np.maximum(0, z)
        else:
            output = np.fromiter((activation(float(value)) for value in z), dtype=float, count=len(z))
        values[node] = output
    return np.asarray(values[network.output_nodes[0]])


def train_eva(experiment, config_path, seed, selectivity, generations=6, n=2_000_000):
    random.seed(seed)
    np.random.seed(seed)
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction, neat.DefaultSpeciesSet, neat.DefaultStagnation, str(config_path))
    if len(config.genome_config.input_keys) != experiment["inputs"].shape[1]:
        raise ValueError("The source configuration and input vector disagree.")
    baseline = evaluate(experiment, baseline_allocation(experiment, n))
    population = []
    next_id = 0
    for _ in range(30):
        genome = neat.DefaultGenome(next_id)
        next_id += 1
        genome.configure_new(config.genome_config)
        population.append(genome)
    best_genome = None
    best_cpr = -np.inf
    history = []
    for generation in range(generations):
        ratios = []
        for genome in population:
            network = neat.nn.FeedForwardNetwork.create(genome, config)
            scores = network_scores(network, experiment["inputs"])
            allocation = rank_allocation(scores, experiment["capacity"], n)
            ratios.append(evaluate(experiment, allocation) / baseline)
        ratios = np.array(ratios)
        cpr = ratios.sum(axis=1)
        for genome, fitness in zip(population, cpr):
            genome.fitness = float(fitness)
        survivors = np.arange(len(population))
        for objective in np.random.permutation(4):
            survivors = survivors[np.argsort(-ratios[survivors, objective], kind="stable")]
            keep = int(np.ceil(len(survivors) * 0.3 ** (selectivity[objective] / selectivity.sum())))
            survivors = survivors[:keep]
        survivors = survivors[np.argsort(-cpr[survivors], kind="stable")][:9]
        winner = int(survivors[0])
        history.append([generation, *ratios[winner], cpr[winner]])
        if cpr[winner] > best_cpr:
            best_cpr = float(cpr[winner])
            best_genome = copy.deepcopy(population[winner])
        if generation + 1 < generations:
            selected = [population[i] for i in survivors]
            pairs = list(itertools.combinations(selected, 2))
            random.shuffle(pairs)
            new_population = [copy.deepcopy(selected[0])]
            for first, second in pairs[:29]:
                child = neat.DefaultGenome(next_id)
                next_id += 1
                child.configure_crossover(first, second, config.genome_config)
                child.mutate(config.genome_config)
                new_population.append(child)
            population = new_population
    network = neat.nn.FeedForwardNetwork.create(best_genome, config)
    return network, pd.DataFrame(history, columns=["Generation", *OBJECTIVES, "CPR"])


def train_with_tuning(experiment, config_path, seed=0, max_rounds=6):
    selectivity = np.ones(4)
    baseline = evaluate(experiment, baseline_allocation(experiment, 2_000_000))
    log = []
    best_network = None
    best_cpr = -np.inf
    best_history = None
    columns = ["Stage", *["Selectivity: " + name for name in OBJECTIVES], *OBJECTIVES]
    for stage in range(max_rounds):
        network, history = train_eva(experiment, config_path, seed + 1000 * stage, selectivity)
        allocation = rank_allocation(network_scores(network, experiment["inputs"]), experiment["capacity"], 2_000_000)
        ratios = evaluate(experiment, allocation) / baseline
        log.append([stage, *selectivity, *ratios])
        if ratios.sum() > best_cpr:
            best_cpr = float(ratios.sum())
            best_network = network
            best_history = history
        if (ratios >= 1).all():
            return network, pd.DataFrame(log, columns=columns), history
        selectivity = np.where(ratios < 1, selectivity / np.maximum(ratios, 1e-9), selectivity)
    return best_network, pd.DataFrame(log, columns=columns), best_history


def eva_projections(experiment, network):
    scores = network_scores(network, experiment["inputs"])
    model_curves = {name: {} for name in OBJECTIVES}
    baseline_curves = {name: {} for name in OBJECTIVES}
    for n in range(100_000, 2_000_001, 100_000):
        allocation = rank_allocation(scores, experiment["capacity"], n)
        baseline = baseline_allocation(experiment, n)
        for name, value, reference in zip(OBJECTIVES, evaluate(experiment, allocation), evaluate(experiment, baseline)):
            model_curves[name][n] = float(value)
            baseline_curves[name][n] = float(reference)
    zip_codes = experiment["data"]["region_name"].to_numpy()
    model = Projection(model_curves, {int(z): int(a) for z, a in zip(zip_codes, allocation) if a > 0}, "Fresh EVA reconstruction")
    baseline = Projection(baseline_curves, {int(z): float(a) for z, a in zip(zip_codes, baseline) if a > 0}, "Reconstructed EIA status quo")
    ratios = pd.DataFrame(model.objective_projections) / pd.DataFrame(baseline.objective_projections)
    ratios.index.name = "Installations"
    return model, baseline, ratios, scores


In [ ]:
raw = pd.read_csv(DATA / "data_by_zip.csv")
eia = pd.read_csv(DATA / "Q4_EIA_March_2024_2025.csv")
experiment = prepare_experiment(raw, eia)
networks = {}
endpoint_rows = []

for seed in [0, 1, 2]:
    network, tuning_log, generation_log = train_with_tuning(experiment, NEAT_CONFIG, seed=seed)
    networks[seed] = network
    model_projection, baseline_projection, ratios, scores = eva_projections(experiment, network)
    endpoint_rows.append([seed, *ratios.loc[2_000_000], int(ratios.ge(1).all(axis=1).sum())])

endpoint_summary = pd.DataFrame(endpoint_rows, columns=["Seed", *OBJECTIVES, "Budgets dominating all objectives"])
print(endpoint_summary.round(4).to_string(index=False))
model_projection, baseline_projection, ratios, scores = eva_projections(experiment, networks[0])


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5.2), layout="constrained")
for name in OBJECTIVES:
    ax.plot(ratios.index / 1_000_000, ratios[name], label=name, linewidth=2)
ax.axhline(1, color="black", linestyle="--")
ax.set(xlabel="Additional installation units (millions)", ylabel="Ratio to reconstructed status quo", title="Figure 3 reconstruction | fresh EVA, seed 0")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


In [ ]:
countries = pd.read_json(DATA / "countries.json", typ="series")
borders = pd.read_json(DATA / "state_lines.json", typ="series")

def draw_us_base(ax, countries, borders):
    for feature in countries['features']:
        if feature['properties'].get('ADM0_A3') == 'USA':
            geometry = feature['geometry']
            polygons = geometry['coordinates'] if geometry['type'] == 'MultiPolygon' else [geometry['coordinates']]
            for polygon in polygons:
                ring = np.asarray(polygon[0])
                ax.fill(ring[:, 0], ring[:, 1], facecolor='#f2f4f6', edgecolor='#78828b', linewidth=0.7, zorder=0)
    for feature in borders['features']:
        if feature['properties'].get('ADM0_A3') == 'USA':
            geometry = feature['geometry']
            lines = geometry['coordinates'] if geometry['type'] == 'MultiLineString' else [geometry['coordinates']]
            for line in lines:
                xy = np.asarray(line)
                ax.plot(xy[:, 0], xy[:, 1], color='#b7bdc4', linewidth=0.5, zorder=1)
    ax.set_aspect(1 / np.cos(np.deg2rad(39)))
    ax.set(xlabel='Longitude', ylabel='Latitude')


data = experiment["data"]
valid = data["Latitude"].between(24, 50) & data["Longitude"].between(-125, -66)
percentiles = pd.Series(scores).rank(pct=True).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5.4), layout="constrained")
for ax, values, label in zip(axes, [scores, percentiles], ["Raw network score", "Score percentile (ranking diagnostic)"]):
    draw_us_base(ax, countries, borders)
    points = ax.scatter(data.loc[valid, "Longitude"], data.loc[valid, "Latitude"], c=values[valid], s=5, cmap="viridis", alpha=0.8, zorder=3)
    fig.colorbar(points, ax=ax, label=label, shrink=0.8)
    ax.set(xlim=(-125, -66), ylim=(24, 50), title=label)
fig.suptitle("Figure 4 reconstruction | ZIP scores, not allocations | seed 0", fontsize=13)
plt.show()


In [ ]:
for n in ratios.index:
    allocation = rank_allocation(scores, experiment["capacity"], int(n))
    baseline = baseline_allocation(experiment, int(n))
    assert allocation.sum() == n
    assert np.isclose(baseline.sum(), n)
    assert (allocation >= 0).all() and (allocation <= experiment["capacity"]).all()
    assert (baseline >= 0).all() and (baseline <= experiment["capacity"] + 1e-6).all()
    equity = evaluate(experiment, allocation)[2:]
    assert ((equity >= 0) & (equity <= 1)).all()

native_scores = np.array([networks[0].activate(row)[0] for row in experiment["inputs"]])
assert np.allclose(native_scores, scores, atol=1e-14, rtol=0)
assert np.array_equal(rank_allocation(native_scores, experiment["capacity"], 2_000_000), rank_allocation(scores, experiment["capacity"], 2_000_000))
print("Budget, capacity, equity bounds, and native-inference checks passed.")


In [ ]:
import pickle
from Simulation.projections_util import create_paper_objectives

released_path = SUNSIGHT / "Models/Neat/models/05-31-25/NEAT_model_lexicase_weighted_5.pkl"
with released_path.open("rb") as stream:
    released_network = pickle.load(stream)

released_scores = network_scores(released_network, experiment["inputs"])
released_allocation = rank_allocation(released_scores, experiment["capacity"], 2_000_000)
reference = baseline_allocation(experiment, 2_000_000)
print("Released checkpoint ratios using the paper-equation reconstruction:")
print(pd.Series(evaluate(experiment, released_allocation) / evaluate(experiment, reference), index=OBJECTIVES).round(4))

zip_codes = experiment["data"]["region_name"].to_numpy()
placed = {int(z): float(a) for z, a in zip(zip_codes, released_allocation) if a > 0}
baseline_placed = {int(z): float(a) for z, a in zip(zip_codes, reference) if a > 0}
print("Native toolkit metric ratios for the same allocations:")
for objective in create_paper_objectives():
    print(objective.name, objective.calc(experiment["data"], placed) / objective.calc(experiment["data"], baseline_placed))
